In [5]:
from pathlib import Path
import shutil

def already_copied(src: Path, dest_dir: Path) -> bool:
    """Return True if src appears to already be copied into dest_dir."""
    src_size = src.stat().st_size

    # 1) Exact same filename already exists (and looks identical by size)
    exact = dest_dir / src.name
    if exact.exists() and exact.stat().st_size == src_size:
        return True

    # 2) Any copy-variant exists with same size (e.g., name_copy1.avi)
    for cand in dest_dir.glob(f"{src.stem}_copy*{src.suffix}"):
        if cand.is_file() and cand.stat().st_size == src_size:
            return True

    return False


def copy_videos_by_keyword(
    folder_path: str,
    keyword: str,
    dest_folder_name: str,
    ext: str = ".avi",
) -> None:
    src_dir = Path(folder_path).expanduser().resolve()
    if not src_dir.exists() or not src_dir.is_dir():
        raise ValueError(f"Not a valid folder: {src_dir}")

    dest_dir = src_dir / dest_folder_name
    dest_dir.mkdir(exist_ok=True)

    ext_l = ext.lower()
    key_l = keyword.lower()

    matches = []
    for p in src_dir.rglob(f"*{ext}"):
        # Do NOT include files that are already inside the destination folder
        if dest_dir in p.parents:
            continue

        if p.suffix.lower() == ext_l and key_l in p.name.lower():
            matches.append(p)

    if not matches:
        print(f"No '{ext}' files containing '{keyword}' found in: {src_dir}")
        return

    copied = 0
    skipped = 0
    renamed = 0

    for src in matches:
        # ---- don't copy again ----
        if already_copied(src, dest_dir):
            skipped += 1
            continue

        dest = dest_dir / src.name

        # If a file with the same name already exists (but seems different), create a new unique name
        if dest.exists():
            stem = src.stem
            suffix = src.suffix
            i = 1
            while True:
                candidate = dest_dir / f"{stem}_copy{i}{suffix}"
                if not candidate.exists():
                    dest = candidate
                    renamed += 1
                    break
                i += 1

        shutil.copy2(src, dest)
        copied += 1

    print(
        f"Searched: {src_dir}\n"
        f"Keyword: '{keyword}' | Extension: '{ext}'\n"
        f"Destination: {dest_dir}\n"
        f"Matched: {len(matches)} | Copied: {copied} | Skipped (already copied): {skipped} | Renamed: {renamed}"
    )


# ---- Run this part ----
folder_path = r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\observationSessions\observationSessions"

copy_videos_by_keyword(
    folder_path=folder_path,
    keyword="face",
    dest_folder_name="trainingSessions_face_facemap",
    ext=".avi",
)




Searched: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\observationSessions\observationSessions
Keyword: 'face' | Extension: '.avi'
Destination: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\observationSessions\observationSessions\trainingSessions_face_facemap
Matched: 131 | Copied: 131 | Skipped (already copied): 0 | Renamed: 0
